# M1 — Beqi-inspired rules-only baseline

M1 isolates the ordered Wolof spelling rules used inside the Eflomal and CMDR pipelines. It performs **no French alignment or substitution**. This makes `M0 → M1 → alignment + M1` an interpretable ablation.

The name *Beqi-inspired* is used because these handwritten expressions are not a complete reproduction of the published Beqi system.

In [1]:
from collections import Counter
import json
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src').exists():
    raise RuntimeError('Run this notebook from the project root or notebooks directory')
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import (
    M1_BEQI_RULES_MANIFEST_PATH, M1_BEQI_RULES_PATH, TRAIN_PARQUET_PATH
)
from src.generation.benchmark_baselines import (
    BEQI_INSPIRED_RULES, METHOD_M1, build_beqi_rule_pairs,
    load_formal_sources, write_dataset_and_manifest,
)

## Configuration

Structural protection of uppercase acronyms and numbers is always enabled. Optional multilingual NER is disabled by default so the notebook runs without a model download. If NER is enabled here, use the same setting for Eflomal and CMDR to keep the comparison controlled.

In [2]:
INPUT_PATH = TRAIN_PARQUET_PATH
OUTPUT_PATH = M1_BEQI_RULES_PATH
MANIFEST_PATH = M1_BEQI_RULES_MANIFEST_PATH
USE_NER = True
NER_MODEL = 'Davlan/xlm-roberta-large-ner-hrl'
NER_BATCH_SIZE = 32
OVERWRITE = True

print(f'Rules:    {len(BEQI_INSPIRED_RULES)} ordered expressions')
print(f'Use NER:  {USE_NER}')
print(f'Output:   {OUTPUT_PATH}')

Rules:    27 ordered expressions
Use NER:  True
Output:   C:\Users\thioy\Desktop\School\PFE\data\synthetic\m1_beqi_rules_pairs.csv


## Load the same formal source pool as M0

In [3]:
sources = load_formal_sources(INPUT_PATH)
print(f'Formal source rows: {len(sources):,}')
sources.head()

Formal source rows: 17,777


,source_id,source_row,formal_wolof
0,parallel:0,0,Bataaxal bii jëwriñu lu ajju ci mbiru bitim ré...
1,parallel:1,1,"""Amuñu woon benn jot ngir rawal sunu bopp."
2,parallel:2,2,"Ekost sa dëkk la te bëggna nu nga toog fi."""
3,parallel:3,3,Xibaari Jotna : Espaañ joxe na juróom-ñett-fuk...
4,parallel:4,4,ñaata at nga am


## Optional named-entity spans

This cell downloads and runs the NER model only when `USE_NER=True`. The generated mapping contains character spans keyed by stable source ID.

In [4]:
entity_spans_by_source = {}
if USE_NER:
    import torch
    from transformers import pipeline

    device = 0 if torch.cuda.is_available() else -1
    ner = pipeline(
        'token-classification', model=NER_MODEL,
        aggregation_strategy='simple', device=device,
    )
    predictions = ner(
        sources.formal_wolof.tolist(), batch_size=NER_BATCH_SIZE
    )
    entity_spans_by_source = {
        source_id: [(entity['start'], entity['end']) for entity in entities]
        for source_id, entities in zip(sources.source_id, predictions)
    }
    print(f'NER spans created for {len(entity_spans_by_source):,} rows')
else:
    print('NER disabled; protecting uppercase acronyms and numbers only.')

c:\Users\thioy\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\cuda\__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: Davlan/xlm-roberta-large-ner-hrl
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


NER spans created for 17,777 rows


## Generate the standalone M1 corpus

In [5]:
m1 = build_beqi_rule_pairs(sources, entity_spans_by_source)

assert len(m1) == len(sources)
assert m1.source_id.is_unique
assert m1.formal_wolof.notna().all() and m1.informal_wolof.notna().all()
assert (m1.formal_wolof.str.len() > 0).all()
assert (m1.informal_wolof.str.len() > 0).all()
assert m1.source_id.tolist() == sources.source_id.tolist()

m1.head(10)

,source_id,source_row,source_corpus,method,seed,informal_wolof,formal_wolof,changed,edit_count,error_types,applied_rules
0,parallel:0,0,french_wolof_parallel,m1_beqi_inspired_rules,2026,batakhal bi jeuwrignou lou ajjou thi mbirou bi...,Bataaxal bii jëwriñu lu ajju ci mbiru bitim ré...,True,22,"[""a_diacritic_to_a"", ""c_vowel_to_th"", ""ee_acut...","{""a_diacritic_to_a"": 1, ""c_vowel_to_th"": 2, ""e..."
1,parallel:1,1,french_wolof_parallel,m1_beqi_inspired_rules,2026,"""amougnou wone benne diot nguir rawal sounou b...","""Amuñu woon benn jot ngir rawal sunu bopp.",True,11,"[""enye_to_gn"", ""final_n_to_ne"", ""j_eao_to_di"",...","{""enye_to_gn"": 1, ""final_n_to_ne"": 2, ""j_eao_t..."
2,parallel:2,2,french_wolof_parallel,m1_beqi_inspired_rules,2026,"ekost sa deukk la té beugna nou nga tog fi.""","Ekost sa dëkk la te bëggna nu nga toog fi.""",True,7,"[""double_g_to_g"", ""final_e_to_e_acute"", ""lower...","{""double_g_to_g"": 1, ""final_e_to_e_acute"": 1, ..."
3,parallel:3,3,french_wolof_parallel,m1_beqi_inspired_rules,2026,Xibaari Jotna : Espaañ diokhé na jouroom gnett...,Xibaari Jotna : Espaañ joxe na juróom-ñett-fuk...,True,19,"[""c_vowel_to_th"", ""enye_to_gn"", ""final_e_to_e_...","{""c_vowel_to_th"": 1, ""enye_to_gn"": 2, ""final_e..."
4,parallel:4,4,french_wolof_parallel,m1_beqi_inspired_rules,2026,gnata at nga am,ñaata at nga am,True,2,"[""enye_to_gn"", ""reduce_double_vowel""]","{""enye_to_gn"": 1, ""reduce_double_vowel"": 1}"
5,parallel:5,5,french_wolof_parallel,m1_beqi_inspired_rules,2026,"Abu Usmaan si, Mamadou Yoro Jallo ak Usmaan nd...","Abu Usmaan Si, Mamadu Yoro Jàllo ak Usmaan Nja...",True,20,"[""a_diacritic_to_a"", ""c_vowel_to_th"", ""enye_to...","{""a_diacritic_to_a"": 2, ""c_vowel_to_th"": 1, ""e..."
6,parallel:6,6,french_wolof_parallel,m1_beqi_inspired_rules,2026,ni gnou ko tamm a wakhé fi : bo khamatoul fo j...,Ni ñu ko tàmm a waxe fii : boo xamatul foo jëm...,True,37,"[""a_diacritic_to_a"", ""enye_to_gn"", ""final_e_to...","{""a_diacritic_to_a"": 1, ""enye_to_gn"": 3, ""fina..."
7,parallel:7,7,french_wolof_parallel,m1_beqi_inspired_rules,2026,"Zambie rew la mou bokk thi ONU, Union Africain...","Zambie réew la mu bokk ci ONU, Union Africaine...",True,3,"[""c_vowel_to_th"", ""ee_acute_to_e"", ""u_to_ou""]","{""c_vowel_to_th"": 1, ""ee_acute_to_e"": 1, ""u_to..."
8,parallel:8,8,french_wolof_parallel,m1_beqi_inspired_rules,2026,toy,tooy,True,1,"[""reduce_double_vowel""]","{""reduce_double_vowel"": 1}"
9,parallel:9,9,french_wolof_parallel,m1_beqi_inspired_rules,2026,"ay temeri nit fatou thia donou bou Nias, thi t...","Ay teemeeri nit faatu ca donu bu Nias, ci tefe...",True,11,"[""c_vowel_to_th"", ""lowercase"", ""reduce_double_...","{""c_vowel_to_th"": 2, ""lowercase"": 1, ""reduce_d..."


## Intrinsic diagnostics

In [6]:
print(f'Rows:           {len(m1):,}')
print(f'Changed rows:   {m1.changed.sum():,} ({m1.changed.mean():.1%})')
print(f'Unchanged rows: {(~m1.changed).sum():,}')
print(f'Duplicate generated strings: {m1.informal_wolof.duplicated().sum():,}')
display(m1.edit_count.describe())

Rows:           17,777
Changed rows:   17,017 (95.7%)
Unchanged rows: 760
Duplicate generated strings: 1,456


count    17777.000000
mean        18.153907
std         16.471553
min          0.000000
25%          6.000000
50%         16.000000
75%         26.000000
max        841.000000
Name: edit_count, dtype: float64

In [7]:
rule_counts = Counter()
for encoded in m1.applied_rules:
    rule_counts.update(json.loads(encoded))
pd.DataFrame(rule_counts.most_common(), columns=['rule', 'applications'])

,rule,applications
0,reduce_double_vowel,65126
1,u_to_ou,62759
2,final_e_to_e_acute,22617
3,c_vowel_to_th,20863
4,final_n_to_ne,20239
5,enye_to_gn,19771
6,x_to_kh,18610
7,schwa_to_eu,15058
8,lowercase,12531
9,soft_g_add_u,12066


In [8]:
m1.loc[m1.changed, [
    'source_id', 'formal_wolof', 'informal_wolof', 'edit_count', 'applied_rules'
]].sample(min(20, int(m1.changed.sum())), random_state=2026)

,source_id,formal_wolof,informal_wolof,edit_count,applied_rules
13346,parallel:13346,Aliw Siise siiwal na altine fan-weeri fan ci w...,aliw sisé siwal na altiné fane weri fane thi w...,25,"{""a_diacritic_to_a"": 1, ""c_vowel_to_th"": 2, ""e..."
4721,parallel:4721,Hichilema mingi bokk ci pàrti United Party for...,hichilema mingui bokk thi parti United Party f...,4,"{""a_diacritic_to_a"": 1, ""c_vowel_to_th"": 1, ""l..."
4554,parallel:4554,wax,wakh,1,"{""x_to_kh"": 1}"
11753,parallel:11753,mbooloo,mbolo,2,"{""reduce_double_vowel"": 2}"
16569,parallel:16569,Jenn table ju ndaw ju weex te rond bu amaale b...,dienne tablé jou ndaw jou wekh té rond bou ama...,25,"{""a_diacritic_to_a"": 1, ""c_vowel_to_th"": 2, ""f..."
17199,parallel:17199,Ay joxekati xibaar yu jokkoo surnaalu koom-koo...,ay diokhekati khibar you diokko sournalou kom ...,37,"{""c_vowel_to_th"": 2, ""double_g_to_g"": 1, ""ee_a..."
13061,parallel:13061,seel,sel,1,"{""reduce_double_vowel"": 1}"
9725,parallel:9725,bàyyi,bayyi,1,"{""a_diacritic_to_a"": 1}"
7316,parallel:7316,Koomu Senegaal dafa wane tagatémbeem ak Korona...,komou Senegaal dafa wané tagatémbem ak Koronaa...,5,"{""final_e_to_e_acute"": 1, ""lowercase"": 1, ""red..."
13079,parallel:13079,"Mu leeral ni Defa saf xorom, defa reetaanlu, n...","mou leral ni defa saf khorom, defa retanlou, n...",21,"{""c_vowel_to_th"": 1, ""double_g_to_g"": 1, ""enye..."


## Export dataset and manifest

Keep unchanged rows in the export: all benchmark methods should use the same formal source pool. Their rate is an intrinsic quality statistic and can later be controlled with an explicitly documented filtered condition.

In [9]:
if not OVERWRITE and (OUTPUT_PATH.exists() or MANIFEST_PATH.exists()):
    raise FileExistsError('M1 output already exists. Set OVERWRITE=True to replace it intentionally.')

manifest = write_dataset_and_manifest(
    m1, OUTPUT_PATH, MANIFEST_PATH, method=METHOD_M1, input_path=INPUT_PATH,
    parameters={
        'rule_count': len(BEQI_INSPIRED_RULES),
        'rule_order_is_significant': True,
        'structural_protection': ['uppercase_acronyms', 'numbers'],
        'use_ner': USE_NER,
        'ner_model': NER_MODEL if USE_NER else None,
        'common_source_pool': True,
    },
)
manifest

{'method': 'm1_beqi_inspired_rules',
 'created_at_utc': '2026-08-03T16:24:56.987805+00:00',
 'input_path': 'C:\\Users\\thioy\\Desktop\\School\\PFE\\data\\translation data\\train-00000-of-00001.parquet',
 'input_sha256': 'bd4910104fc121ff6924ca9a8b1a0d6bdaa6f28601108703cc9b29b2dc9eb27e',
 'output_path': 'C:\\Users\\thioy\\Desktop\\School\\PFE\\data\\synthetic\\m1_beqi_rules_pairs.csv',
 'output_sha256': 'ba6c2d671a40f52d78a7376f4aac193a2cb2a680102abd8e5f3f85d44df54aff',
 'rows': 17777,
 'unique_sources': 17777,
 'changed_rows': 17017,
 'unchanged_rows': 760,
 'parameters': {'rule_count': 27,
  'rule_order_is_significant': True,
  'structural_protection': ['uppercase_acronyms', 'numbers'],
  'use_ner': True,
  'ner_model': 'Davlan/xlm-roberta-large-ner-hrl',
  'common_source_pool': True}}